# EDA — Loan Amount Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)

df = pd.read_csv('../data/raw/credit_risk.csv')
print(f'{df.shape[0]:,} rows × {df.shape[1]} columns')
print(df.dtypes)

## Schema & Summary Stats

In [ ]:
df.describe().T.round(2)

## Missing Values

`loan_int_rate` (9.6%) and `person_emp_length` (2.7%) need imputation — dropping them would lose two of the most useful features.

In [ ]:
missing = df.isnull().sum().sort_values(ascending=False)
miss_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Count': missing, 'Pct': miss_pct}).query('Count > 0')

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(miss_df.index, miss_df['Pct'],
               color=['#C62828' if p>5 else '#FF9800' for p in miss_df['Pct']])
for bar, (col, row) in zip(bars, miss_df.iterrows()):
    ax.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2,
            f"{int(row['Count']):,} ({row['Pct']}%)", va='center', fontsize=10)
ax.set_xlabel('Missing %'); ax.set_xlim(0, 15)
ax.set_title('Missing values by column', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## Target Distribution: `loan_amnt`

All USD. Range $500–$35,000, mean $9,589. Heavily right-skewed — most loans are under $15k.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['loan_amnt'], bins=60, color='#1565C0', edgecolor='white', alpha=0.85)
axes[0].axvline(df['loan_amnt'].mean(),   color='red',    lw=2, linestyle='--', label=f'Mean = ${df["loan_amnt"].mean():,.0f}')
axes[0].axvline(df['loan_amnt'].median(), color='orange', lw=2, linestyle='--', label=f'Median = ${df["loan_amnt"].median():,.0f}')
axes[0].set_title('Loan amount distribution', fontweight='bold')
axes[0].set_xlabel('Loan Amount (USD)'); axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:,.0f}'))
axes[0].legend()

loan_intent_avg = df.groupby('loan_intent')['loan_amnt'].mean().sort_values(ascending=False)
axes[1].bar(loan_intent_avg.index, loan_intent_avg.values/1000, color='#1565C0', edgecolor='white')
axes[1].set_title('Average loan amount by intent', fontweight='bold')
axes[1].set_xlabel('Loan Intent'); axes[1].set_ylabel('Avg ($k)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('../outputs/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Min: ${df["loan_amnt"].min():,}  Max: ${df["loan_amnt"].max():,}  Mean: ${df["loan_amnt"].mean():,.0f}')

## Numeric Features vs Loan Amount

In [ ]:
numeric_cols = ['person_age','person_income','person_emp_length','loan_int_rate','cb_person_cred_hist_length']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].scatter(df[col], df['loan_amnt']/1000, alpha=0.15, s=8, color='#1565C0')
    axes[i].set_xlabel(col); axes[i].set_ylabel('Loan Amount ($k)')
    axes[i].set_title(col, fontweight='bold')
axes[-1].axis('off')
plt.suptitle('No single feature predicts loan amount cleanly', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## Correlations

Highest correlation with `loan_amnt` is income at ~0.34. The features measure creditworthiness, not borrowing intent — this is why R² ends up modest.

In [ ]:
num_for_corr = ['person_age','person_income','person_emp_length','loan_amnt','loan_int_rate','cb_person_cred_hist_length']
corr = df[num_for_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5, annot_kws={'size':11})
ax.set_title('Correlation matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('Correlations with loan_amnt:')
for f, v in corr['loan_amnt'].drop('loan_amnt').sort_values(key=abs, ascending=False).items():
    print(f'  {f:<30} {v:+.3f}')

## Categorical Features

In [ ]:
cat_cols = ['loan_grade','person_home_ownership','loan_intent','cb_person_default_on_file']
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    avg = df.groupby(col)['loan_amnt'].mean().sort_values(ascending=False)
    axes[i].bar(avg.index, avg.values/1000, color='#1565C0', edgecolor='white')
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Avg Loan Amount ($k)')
    axes[i].tick_params(axis='x', rotation=15)
plt.suptitle('Average loan amount by category', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/categorical_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()